# preprocessing text data for llms 

In [ ]:
# Save the dataset to disk
    from datasets import load_from_disk

    dataset = load_from_disk(
        r"C:\LLM from Scratch\datasets\tiny_stories_dataset"
    )

In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

### createing a small dataset and trying to split it using `re`  

In [8]:
import re
text = "This is a` sample text with ,some punctuation! Let's clean it up."
result = re.split(r'[^\w\s`]', text)

print(result)

['This is a` sample text with ', 'some punctuation', ' Let', 's clean it up', '']


In [9]:
result = re.split(r'([,.]|\s)', text)
result  = [word for word in result if word.strip() != '']
print(result)


['This', 'is', 'a`', 'sample', 'text', 'with', ',', 'some', 'punctuation!', "Let's", 'clean', 'it', 'up', '.']


lets apply this kind of tokenizer to dataset

#### our dataset is an list of strings so lets join our all the list into single string and convert it into tokens 

In [10]:
dataset_text = " ".join(dataset["train"]['text'][:20000])  # Join the first 20,000 entries of the 'text' column into a single string

In [11]:
preprocessed_text = re.split(r'([,.:;?_!"()\']|--|\s)', dataset_text)
preprocessed_text = [word for word in preprocessed_text if word.strip() != '']
print(preprocessed_text[:100])

['One', 'day', ',', 'a', 'little', 'girl', 'named', 'Lily', 'found', 'a', 'needle', 'in', 'her', 'room', '.', 'She', 'knew', 'it', 'was', 'difficult', 'to', 'play', 'with', 'it', 'because', 'it', 'was', 'sharp', '.', 'Lily', 'wanted', 'to', 'share', 'the', 'needle', 'with', 'her', 'mom', ',', 'so', 'she', 'could', 'sew', 'a', 'button', 'on', 'her', 'shirt', '.', 'Lily', 'went', 'to', 'her', 'mom', 'and', 'said', ',', '"', 'Mom', ',', 'I', 'found', 'this', 'needle', '.', 'Can', 'you', 'share', 'it', 'with', 'me', 'and', 'sew', 'my', 'shirt', '?', '"', 'Her', 'mom', 'smiled', 'and', 'said', ',', '"', 'Yes', ',', 'Lily', ',', 'we', 'can', 'share', 'the', 'needle', 'and', 'fix', 'your', 'shirt', '.', '"', 'Together']


In [12]:
print(f"Total number of tokens: {len(preprocessed_text)}")

Total number of tokens: 4236944


## converting token into token id

Next, let’s convert these tokens from a Python string to an integer representation to
produce the token IDs. This conversion is an intermediate step before converting the
token IDs into embedding vectors.


let’s create a list of all unique tokens and sort them
alphabetically to determine the vocabulary size:

In [13]:
all_words = sorted(set(preprocessed_text))
vocab_size = len(all_words)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 13395


we have total `13395` unique words among `4236944` tokens 


## creating the Vocabulary

lets print first 100 vocabulary 

In [23]:
all_words.extend(["<|endoftext|>", "<|unk|>"])
vocab = {word: idx for idx, word in enumerate(all_words)}
for i, item in enumerate(list(vocab.items())[:101]):
    print(item)
    if i >= 101:
        break   

('!', 0)
('"', 1)
('$10', 2)
('$500', 3)
('&', 4)
("'", 5)
('(', 6)
(')', 7)
('*beep*', 8)
('*poof*', 9)
(',', 10)
('-', 11)
('--', 12)
('.', 13)
('1', 14)
('10', 15)
('100', 16)
('10th', 17)
('12', 18)
('123', 19)
('15', 20)
('16', 21)
('164', 22)
('2', 23)
('20', 24)
('20kg', 25)
('25', 26)
('29', 27)
('3', 28)
('3-year', 29)
('3-year-old', 30)
('3-years-old', 31)
('30', 32)
('35', 33)
('3rd', 34)
('4', 35)
('40', 36)
('5', 37)
('50', 38)
('6', 39)
('7', 40)
('75', 41)
('76', 42)
('8', 43)
('80', 44)
('911', 45)
(':', 46)
(';', 47)
('?', 48)
('A', 49)
('A+', 50)
('ABC', 51)
('ABCs', 52)
('ACHOO', 53)
('AHOY', 54)
('ALL', 55)
('AND', 56)
('ARE', 57)
('Abbie', 58)
('Abby', 59)
('Abi', 60)
('Abigail', 61)
('Abracadabra', 62)
('Abraham', 63)
('Absolutely', 64)
('Acceptor', 65)
('Accidents', 66)
('Achoo', 67)
('Acorns', 68)
('Across', 69)
('Acting', 70)
('Adam', 71)
('Add', 72)
('Adding', 73)
('Adventure', 74)
('Adventurers', 75)
('Afraid', 76)
('Africa', 77)
('After', 78)
('Afterward', 7

As we can see, the dictionary contains individual tokens associated with unique inte
ger labels. Our next goal is to apply this vocabulary to convert new text into token IDs

## Implementing a simple text tokenizer

In [26]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>"
            for item in preprocessed
        ]
        
        preprocessed = [item if item in self.str_to_int           
                else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)   
        return text

In [27]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," 
       Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 1123, 5, 10044, 11779, 7458, 6683, 8662, 10, 13028, 7404, 10, 1, 1576, 13, 13400, 10066, 12862, 13400, 9271, 13]


Next, let’s see whether we can turn these token IDs back into text using the decode
method

In [28]:
# convert the ids back to text
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. <|unk|> said with <|unk|> pride.


lets try new text that are not in dataset

In [31]:
name = "lalu prajapati is a machine learning engineer"
print(tokenizer.encode(name))

[13400, 13400, 7189, 2732, 7772, 7518, 5483]


converting ids to their orignal tokens 

In [32]:
print(tokenizer.decode(tokenizer.encode(name)))

<|unk|> <|unk|> is a machine learning engineer


# Byte pair encoding